# 01 — Construção do dataset: reeleição de prefeitos (2020 e 2024)

**Projeto final — Inteligência Artificial (UFPI)**

Este notebook monta a base usada na modelagem. Ele **não treina modelos**: apenas coleta, limpa e cruza dados públicos.

**Fonte:** [Base dos Dados](https://basedosdados.org), que redistribui os dados abertos do
[TSE](https://dadosabertos.tse.jus.br) já padronizados no Google BigQuery. A consulta exige um projeto
gratuito no Google Cloud (modo *sandbox*).

**Unidade de análise:** um prefeito eleito em uma eleição ordinária (2016 ou 2020) que **disputou a
reeleição** no mesmo município na eleição seguinte (2020 ou 2024).

**Privacidade (LGPD):** a tabela de candidatos do TSE contém CPF, título de eleitor e nome. O título é
usado **apenas internamente**, para ligar a mesma pessoa entre duas eleições, e é descartado antes de
salvar a base. Nenhum dado pessoal identificável é exibido ou publicado.

## 1. Configuração

Autentica a conta Google, cria a conexão com o BigQuery e define os parâmetros do estudo.
Para incluir outra eleição (por exemplo, 2016), basta acrescentá-la nos dicionários abaixo.

In [1]:
from google.colab import auth
auth.authenticate_user()                       # janela de permissão do Google

from google.cloud import bigquery
import pandas as pd
import numpy as np
import unicodedata
import re

ID_PROJETO = "ia-reeleicao"                    # projeto no Google Cloud que "paga" as consultas (100% gratuito)
client = bigquery.Client(project=ID_PROJETO)
TSE = "basedosdados.br_tse_eleicoes"           # conjunto de tabelas do TSE na Base dos Dados

def consultar(sql):
    # Envia uma consulta SQL ao BigQuery e devolve o resultado como DataFrame do pandas.
    return client.query(sql).to_dataframe()

# Parâmetros do estudo
ELEICOES = [2020, 2024]                        # eleições em que o prefeito tenta a reeleição (analisamos essas eleições)
ANO_ANTERIOR = {2020: 2016, 2024: 2020}        # eleição em que esse prefeito foi eleito (filtramos os possíveis candidatos a partir daí)
ANO_GOVERNADOR = {2020: 2018, 2024: 2022}      # eleição do governador em exercício (serve para atrelarmos situação/oposição do prefeito em relação ao governo do estado)
ANOS_MUNICIPAIS = sorted(set(ELEICOES) | set(ANO_ANTERIOR.values()))   # [2016, 2020, 2024]

lista = lambda anos: ", ".join(str(a) for a in anos)   # [2016, 2020] -> "2016, 2020" (para o SQL)
FILTRO_ORDINARIA = "tipo_eleicao = 'eleicao ordinaria'"  # exclui eleições suplementares
RESULTADO_ELEITO = "eleito"                    # valor da coluna 'resultado' para quem venceu (conferir no diagnóstico)

## 2. Diagnóstico rápido

Antes da construção da amostra, os valores que aparecem na coluna `resultado` e o tipo de
eleição em cada ano são conferidos. Isso garante que os filtros usados adiante (eleição ordinária e "eleito") existem
em todos os anos.

In [2]:
diag = consultar(f'''
SELECT ano, tipo_eleicao, turno, resultado, COUNT(*) AS n
FROM `{TSE}.resultados_candidato_municipio`
WHERE ano IN ({lista(ANOS_MUNICIPAIS)}) AND cargo = 'prefeito'
  AND LOWER(tipo_eleicao) NOT LIKE '%suplementar%'
GROUP BY ano, tipo_eleicao, turno, resultado
ORDER BY ano, turno, n DESC
''')
print(diag.to_string())

     ano                           tipo_eleicao  turno   resultado      n
0   2016                      eleicao ordinaria      1  nao eleito   9985
1   2016                      eleicao ordinaria      1      eleito   5460
2   2016                      eleicao ordinaria      1    2º turno    114
3   2016       eleicao sup santa luzia do norte      1  nao eleito      5
4   2016   nova eleicao set/2017 - abelardo luz      1  nao eleito      3
5   2016  eleicao suplem sao jose da bela vista      1  nao eleito      2
6   2016     nova eleicao mar/2020 - laurentino      1  nao eleito      2
7   2016     eleicao suplement bela vista do ma      1  nao eleito      2
8   2016               alto taquari-mt-jul-2017      1  nao eleito      2
9   2016               alto taquari-mt-jul-2017      1      eleito      1
10  2016   eleicao sup de sao jose de campestre      1  nao eleito      1
11  2016      eleicao sup bom jesus do araguaia      1      eleito      1
12  2016          sup de serranopolis 

## 3. Coleta dos dados do TSE

São três consultas, todas restritas ao cargo de prefeito e às eleições ordinárias:

1. **Candidatos**: partido, situação da candidatura, idade, gênero e título de eleitor (chave da pessoa).
2. **Resultados por município**: votos e resultado de cada candidato em cada turno.
3. **Governadores eleitos** em 2018 e 2022, para medir o alinhamento político do prefeito.

In [3]:
cand = consultar(f'''
SELECT ano, sigla_uf, id_municipio, id_municipio_tse, sequencial, titulo_eleitoral,
       sigla_partido, situacao, genero, idade
FROM `{TSE}.candidatos`
WHERE ano IN ({lista(ANOS_MUNICIPAIS)}) AND cargo = 'prefeito' AND {FILTRO_ORDINARIA}
''')

res = consultar(f'''
SELECT ano, turno, id_municipio_tse, sequencial_candidato AS sequencial, resultado, votos
FROM `{TSE}.resultados_candidato_municipio`
WHERE ano IN ({lista(ANOS_MUNICIPAIS)}) AND cargo = 'prefeito' AND {FILTRO_ORDINARIA}
''')

gov = consultar(f'''
SELECT DISTINCT ano, sigla_uf, sigla_partido
FROM `{TSE}.resultados_candidato_municipio`
WHERE ano IN ({lista(ANO_GOVERNADOR.values())}) AND cargo = 'governador'
  AND {FILTRO_ORDINARIA} AND resultado = '{RESULTADO_ELEITO}'
''')

# Os códigos vêm às vezes como número, às vezes como texto: padronizamos tudo como texto
# para que os merges entre tabelas funcionem.
for df in (cand, res):
    df["id_municipio_tse"] = df["id_municipio_tse"].astype(str)
    df["sequencial"] = df["sequencial"].astype(str)
cand["titulo_eleitoral"] = cand["titulo_eleitoral"].astype(str)
res["votos"] = pd.to_numeric(res["votos"], errors="coerce").fillna(0)

print("Candidatos por ano:", cand.groupby("ano").size().to_dict())
print("Linhas de resultado por ano:", res.groupby("ano").size().to_dict())
print("Governadores eleitos por ano:", gov.groupby("ano").size().to_dict(), "(esperado: 27 por ano)")

Candidatos por ano: {np.int64(2016): 16546, np.int64(2020): 19264, np.int64(2024): 15547}
Linhas de resultado por ano: {np.int64(2016): 15673, np.int64(2020): 18227, np.int64(2024): 15151}
Governadores eleitos por ano: {np.int64(2018): 27, np.int64(2022): 27} (esperado: 27 por ano)


## 4. Votos no 1º turno

Para cada candidato calculamos, no 1º turno:

- `pct_votos`: proporção dos votos do município obtida pelo candidato;
- `margem`: diferença entre o seu percentual e o do melhor adversário;
- `n_candidatos`: número de candidatos que receberam votos no município.

Usamos o **1º turno** porque ele é único na grande maioria das cidades do país; o 2º turno só ocorre em cidades com mais
de 200 mil eleitores, e dessas cidades, boa parte finaliza as eleições ainda em primeiro turno. Candidatos com votos anulados (candidatura indeferida) ficam com zero votos e não
contam como adversários.

In [4]:
t1 = res[res["turno"] == 1].copy()
chave_mun = ["ano", "id_municipio_tse"]

t1["votos_mun"] = t1.groupby(chave_mun)["votos"].transform("sum")
t1["pct_votos"] = t1["votos"] / t1["votos_mun"].replace(0, np.nan)

# número de candidatos que receberam votos
t1["n_candidatos"] = t1.groupby(chave_mun)["votos"].transform(lambda v: (v > 0).sum())

# 1º e 2º maiores percentuais do município, para calcular a margem de cada candidato
def segundo_maior(s):
    valores = s.nlargest(2).tolist()
    return valores[1] if len(valores) > 1 else 0.0

topo = t1.groupby(chave_mun)["pct_votos"].agg(p1="max", p2=segundo_maior).reset_index()
t1 = t1.merge(topo, on=chave_mun)
t1["margem"] = np.where(t1["pct_votos"] >= t1["p1"],
                        t1["pct_votos"] - t1["p2"],      # 1º colocado: distância para o 2º
                        t1["pct_votos"] - t1["p1"])      # demais: distância para o 1º (negativa)

# junta o título de eleitor, o partido e a situação do candidato
t1 = t1.merge(cand, on=["ano", "id_municipio_tse", "sequencial"], how="left")
print("Linhas do 1º turno sem correspondência na tabela de candidatos:", t1["titulo_eleitoral"].isna().sum())

# vencedores de cada eleição (em qualquer turno)
eleitos = (res[res["resultado"] == RESULTADO_ELEITO][["ano", "id_municipio_tse", "sequencial"]]
           .drop_duplicates()
           .merge(cand, on=["ano", "id_municipio_tse", "sequencial"], how="left"))
print("Prefeitos eleitos por ano:", eleitos.groupby("ano").size().to_dict())

Linhas do 1º turno sem correspondência na tabela de candidatos: 10
Prefeitos eleitos por ano: {np.int64(2016): 5517, np.int64(2020): 5495, np.int64(2024): 5519}


## 5. Montagem da amostra e critérios de exclusão

Para cada eleição *t* (2020 e 2024):

1. pegamos os prefeitos eleitos em *t − 4*;
2. verificamos, pelo **título de eleitor**, quem registrou candidatura a prefeito no **mesmo município** em *t*;
3. excluímos candidaturas que não chegaram a disputar de fato (indeferida, renúncia, cassada etc.);
4. exigimos que o candidato apareça na apuração do 1º turno;
5. excluímos **candidaturas únicas** (sem adversário com votos): nesses casos o resultado é
   determinado por construção e não há o que prever.

Esses casos **não são outliers**: são critérios que definem a população do estudo. Eleições suplementares
ficam de fora pelo filtro de eleição ordinária, e vices que assumiram o cargo durante o mandato também
ficam de fora, porque não foram eleitos como prefeitos em *t − 4*.

O alvo é `reeleito` = 1 se o prefeito venceu a eleição *t* (em qualquer turno).

In [5]:
SITUACOES_EXCLUIDAS = {
    "indeferido", "indeferido com recurso", "renuncia", "cassado", "cassado com recurso",
    "falecido", "cancelado", "pedido nao conhecido", "pedido nao conhecido com recurso",
    "pendente de julgamento",
}

partes, fluxo = [], []
for t in ELEICOES:
    ant = ANO_ANTERIOR[t]

    # (1) eleitos em t-4, com os dados daquela eleição (sufixo _ant)
    el = eleitos[eleitos["ano"] == ant][["id_municipio_tse", "titulo_eleitoral", "sequencial", "sigla_partido"]]
    el = el.rename(columns={"sequencial": "sequencial_ant", "sigla_partido": "partido_ant"})
    votos_ant = t1[t1["ano"] == ant][["id_municipio_tse", "sequencial", "pct_votos", "margem"]]
    votos_ant = votos_ant.rename(columns={"sequencial": "sequencial_ant", "pct_votos": "pct_votos_anterior",
                                          "margem": "margem_anterior"})
    el = el.merge(votos_ant, on=["id_municipio_tse", "sequencial_ant"], how="left")
    n0 = len(el)

    # (2) registraram candidatura a prefeito no mesmo município em t
    cand_t = cand[cand["ano"] == t].drop(columns="ano")
    inc = el.merge(cand_t, on=["id_municipio_tse", "titulo_eleitoral"], how="inner")
    n1 = len(inc)

    # (3) situação da candidatura válida
    inc = inc[~inc["situacao"].isin(SITUACOES_EXCLUIDAS)]
    n2 = len(inc)

    # (4) aparecem na apuração do 1º turno, com votos
    votos_t = t1[t1["ano"] == t][["id_municipio_tse", "sequencial", "votos", "pct_votos",
                                   "margem", "n_candidatos"]]
    inc = inc.merge(votos_t, on=["id_municipio_tse", "sequencial"], how="inner")
    inc = inc[inc["votos"] > 0]
    n3 = len(inc)

    # (5) disputa competitiva: pelo menos dois candidatos com votos
    inc = inc[inc["n_candidatos"] >= 2]
    n4 = len(inc)

    # alvo: venceu a eleição t?
    venc_t = set(eleitos[eleitos["ano"] == t]["sequencial"] + "|" + eleitos[eleitos["ano"] == t]["id_municipio_tse"])
    inc["reeleito"] = (inc["sequencial"] + "|" + inc["id_municipio_tse"]).isin(venc_t).astype(int)

    inc["ano"] = t
    partes.append(inc)
    fluxo.append({"eleição": t, "eleitos em t-4": n0, "registraram candidatura": n1,
                  "situação válida": n2, "votados no 1º turno": n3, "disputa competitiva (amostra final)": n4,
                  "taxa de reeleição na amostra": round(inc["reeleito"].mean(), 3)})

amostra = pd.concat(partes, ignore_index=True)
fluxo = pd.DataFrame(fluxo).set_index("eleição").T
fluxo

eleição,2020,2024
eleitos em t-4,5517.000,5495.000
registraram candidatura,3221.000,2833.000
situação válida,3107.000,2809.000
votados no 1º turno,3104.000,2780.000
disputa competitiva (amostra final),2998.000,2624.000
taxa de reeleição na amostra,0.643,0.829


**Conferência:** segundo levantamentos divulgados a partir de dados do TSE, a taxa de reeleição entre os
prefeitos que tentaram foi de cerca de 63% em 2020 e de 81–82% em 2024. A taxa da nossa amostra deve
ficar próxima desses valores (com pequenas diferenças porque excluímos as candidaturas únicas).

## 6. Atributos do bloco político

- `troca_partido`: o prefeito disputou a reeleição por um partido diferente daquele pelo qual foi eleito;
- `mesmo_partido_governador`: o partido do prefeito é o mesmo do governador do estado em exercício;
- `coligacao_com_governador`: o partido do governador integra a coligação do prefeito.

Partidos mudam de nome e se fundem (PR → PL, DEM + PSL → União etc.). Para comparar siglas de anos
diferentes, convertemos cada sigla para o nome **vigente no ano da eleição analisada**. A tabela de
sucessão foi reaproveitada de um trabalho anterior em grupo do qual o autor participou (repositório
[analiseCandidatos](https://github.com/SoaresDavidson/analiseCandidatos), trabalho em grupo da disciplina
de Banco de Dados).

In [6]:
# (sigla antiga, sigla nova, ano da mudança)
SUCESSAO = [
    ("PMDB", "MDB", 2017), ("PTN", "PODE", 2017), ("PT do B", "AVANTE", 2017), ("SD", "SOLIDARIEDADE", 2017),
    ("PSDC", "DC", 2018), ("PEN", "PATRIOTA", 2018), ("PR", "PL", 2019), ("PRB", "REPUBLICANOS", 2019),
    ("PPS", "CIDADANIA", 2019), ("PHS", "PODE", 2019), ("PRP", "PATRIOTA", 2019), ("PPL", "PC do B", 2019),
    ("DEM", "UNIÃO", 2022), ("PSL", "UNIÃO", 2022), ("PTC", "AGIR", 2022), ("PROS", "SOLIDARIEDADE", 2023),
    ("PSC", "PODE", 2023), ("PTB", "PRD", 2023), ("PATRIOTA", "PRD", 2023), ("PMN", "MOBILIZA", 2023),
]

def normalizar(sigla):
    # 'PC do B' -> 'PCDOB'; 'UNIÃO' -> 'UNIAO' (sem acento, espaço ou pontuação, em maiúsculas)
    if sigla is None or (isinstance(sigla, float) and np.isnan(sigla)):
        return None
    s = unicodedata.normalize("NFKD", str(sigla)).encode("ascii", "ignore").decode()
    s = re.sub(r"[^A-Z0-9]", "", s.upper())
    return {"PODEMOS": "PODE", "UNIAOBRASIL": "UNIAO"}.get(s, s)

SUCESSAO_NORM = [(normalizar(a), normalizar(b), ano) for a, b, ano in SUCESSAO]

def sigla_no_ano(sigla, ano):
    # Converte a sigla para o nome que o partido tinha no ano informado (aplica fusões em cadeia).
    s = normalizar(sigla)
    mudou = True
    while mudou and s is not None:
        mudou = False
        for antiga, nova, ano_mudanca in SUCESSAO_NORM:
            if s == antiga and ano_mudanca <= ano:
                s, mudou = nova, True
    return s

# --- troca de partido ---
amostra["troca_partido"] = [
    int(sigla_no_ano(p_ant, t) != sigla_no_ano(p_t, t))
    for p_ant, p_t, t in zip(amostra["partido_ant"], amostra["sigla_partido"], amostra["ano"])
]

# --- governador em exercício na eleição t ---
gov["ano_municipal"] = gov["ano"].map({v: k for k, v in ANO_GOVERNADOR.items()})
gov_partido = {(r.ano_municipal, r.sigla_uf): r.sigla_partido for r in gov.itertuples()}

amostra["partido_governador"] = [gov_partido.get((t, uf)) for t, uf in zip(amostra["ano"], amostra["sigla_uf"])]
amostra["mesmo_partido_governador"] = [
    int(g is not None and sigla_no_ano(p, t) == sigla_no_ano(g, t))
    for p, g, t in zip(amostra["sigla_partido"], amostra["partido_governador"], amostra["ano"])
]
print("Prefeitos sem governador identificado:", amostra["partido_governador"].isna().sum())

Prefeitos sem governador identificado: 0


In [7]:
# --- coligação do prefeito na eleição t ---
part = consultar(f'''
SELECT DISTINCT ano, id_municipio_tse, sigla, sequencial_coligacao, composicao_coligacao
FROM `{TSE}.partidos`
WHERE ano IN ({lista(ELEICOES)}) AND cargo = 'prefeito' AND {FILTRO_ORDINARIA}
''')
part["id_municipio_tse"] = part["id_municipio_tse"].astype(str)
part["sigla_norm"] = part["sigla"].map(normalizar)
print("Exemplos de composição de coligação:")
print(part["composicao_coligacao"].dropna().head(5).to_string())

# índice: (ano, município, sigla) -> (sequencial da coligação, composição em texto)
idx_partido = {(r.ano, r.id_municipio_tse, r.sigla_norm): (r.sequencial_coligacao, r.composicao_coligacao)
               for r in part.itertuples()}
# índice: (ano, município, sequencial da coligação) -> siglas dos partidos dessa coligação
membros = part.dropna(subset=["sequencial_coligacao"]).groupby(
    ["ano", "id_municipio_tse", "sequencial_coligacao"])["sigla_norm"].apply(set).to_dict()

def partidos_da_coligacao(ano, mun, sigla):
    # Conjunto de partidos que apoiaram o prefeito: a própria sigla + os demais da coligação.
    s = normalizar(sigla)
    conjunto = {s}
    seq, comp = idx_partido.get((ano, mun, s), (None, None))
    if seq is not None and not pd.isna(seq):
        conjunto |= membros.get((ano, mun, seq), set())
    if isinstance(comp, str):                      # texto do tipo "PT / PSB / PC do B"
        conjunto |= {normalizar(x) for x in comp.split("/") if x.strip()}
    return conjunto

amostra["n_partidos_coligacao"] = 0
amostra["coligacao_com_governador"] = 0
for i, r in amostra.iterrows():
    colig = partidos_da_coligacao(r["ano"], r["id_municipio_tse"], r["sigla_partido"])
    colig_no_ano = {sigla_no_ano(x, r["ano"]) for x in colig if x}
    amostra.at[i, "n_partidos_coligacao"] = len(colig_no_ano)
    if r["partido_governador"] is not None:
        amostra.at[i, "coligacao_com_governador"] = int(sigla_no_ano(r["partido_governador"], r["ano"]) in colig_no_ano)

amostra.groupby("ano")[["troca_partido", "mesmo_partido_governador", "coligacao_com_governador"]].mean().round(3)

Exemplos de composição de coligação:
0    AGIR
1    AGIR
2    AGIR
3    AGIR
4    AGIR


,troca_partido,mesmo_partido_governador,coligacao_com_governador
ano,,,
2020,0.360,0.165,0.307
2024,0.484,0.245,0.473


## 7. Gasto de campanha (bloco "dinheiro")

Somamos as despesas de campanha (apenas as declaradas ao TSE) de cada candidato a prefeito e calculamos a
**participação do prefeito no gasto total** do município (`gasto_campanha_share`). A soma é feita no
próprio BigQuery, então só o resultado agregado é baixado.

**Atenção para a interpretação:** gasto de campanha não é causa da vitória de forma direta. Candidatos com
mais chance de vencer também atraem mais recursos (causalidade reversa). No modelo, ele é tratado
apenas como informação preditiva.

In [8]:
desp = consultar(f'''
SELECT ano, id_municipio_tse, sequencial_candidato AS sequencial, SUM(valor_despesa) AS gasto
FROM `{TSE}.despesas_candidato`
WHERE ano IN ({lista(ELEICOES)}) AND cargo = 'prefeito' AND {FILTRO_ORDINARIA}
GROUP BY ano, id_municipio_tse, sequencial
''')
desp["id_municipio_tse"] = desp["id_municipio_tse"].astype(str)
desp["sequencial"] = desp["sequencial"].astype(str)
desp["gasto_total_mun"] = desp.groupby(["ano", "id_municipio_tse"])["gasto"].transform("sum")
print("Candidatos com despesas declaradas por ano:", desp.groupby("ano").size().to_dict())

amostra = amostra.merge(desp[["ano", "id_municipio_tse", "sequencial", "gasto", "gasto_total_mun"]],
                        on=["ano", "id_municipio_tse", "sequencial"], how="left")
amostra["gasto_campanha"] = amostra["gasto"].fillna(0)
amostra["gasto_campanha_share"] = amostra["gasto"] / amostra["gasto_total_mun"].replace(0, np.nan)
print("Prefeitos sem despesa declarada:", amostra["gasto"].isna().sum())

Candidatos com despesas declaradas por ano: {np.int64(2020): 19178, np.int64(2024): 15435}
Prefeitos sem despesa declarada: 1


## 8. Base final do TSE

Mantemos apenas os códigos do município (para cruzar com as outras fontes), o alvo e os atributos.
**Título de eleitor, sequencial e siglas partidárias são descartados** (privacidade e neutralidade: o
modelo não usa a identidade do partido). A margem de votos em *t* (`alvo_margem`) é guardada só como
alvo alternativo: **não pode ser usada como atributo**, pois é o próprio resultado da eleição.

In [9]:
amostra["genero_feminino"] = (amostra["genero"].str.lower() == "feminino").astype(int)
amostra = amostra.rename(columns={"margem": "alvo_margem"})

colunas_finais = [
    "ano", "sigla_uf", "id_municipio", "id_municipio_tse", "reeleito", "alvo_margem",
    # contexto político
    "pct_votos_anterior", "margem_anterior", "n_candidatos", "troca_partido",
    "mesmo_partido_governador", "coligacao_com_governador", "n_partidos_coligacao",
    "idade", "genero_feminino",
    # dinheiro (campanha)
    "gasto_campanha", "gasto_campanha_share",
]
base_tse = amostra[colunas_finais].copy()
base_tse.to_csv("base_tse.csv", index=False)

print(base_tse.shape)
base_tse.describe().T.round(3)

(5622, 17)


,count,mean,std,min,25%,50%,75%,max
ano,5622.0,2021.866951,1.995747,2020.0,2020.0,2020.0,2024.0,2024.0
reeleito,5622.0,0.729634,0.444189,0.0,0.0,1.0,1.0,1.0
alvo_margem,5622.0,0.175789,0.267746,-0.787733,-0.012961,0.140789,0.348895,0.996139
pct_votos_anterior,5622.0,0.533879,0.122451,0.154541,0.475196,0.526443,0.576798,1.0
margem_anterior,5622.0,0.164781,0.186077,-0.225307,0.049544,0.111797,0.208891,1.0
n_candidatos,5622.0,3.03842,1.493489,2.0,2.0,3.0,3.0,16.0
troca_partido,5622.0,0.417823,0.493245,0.0,0.0,0.0,1.0,1.0
mesmo_partido_governador,5622.0,0.202419,0.401839,0.0,0.0,0.0,0.0,1.0
coligacao_com_governador,5622.0,0.384205,0.48645,0.0,0.0,0.0,1.0,1.0
n_partidos_coligacao,5622.0,4.235859,2.61791,1.0,2.0,4.0,6.0,20.0


## 9. Indicadores do mandato: finanças, economia e educação

Para cada eleição *t*, os indicadores do mandato usam **apenas anos anteriores à eleição**: de *t − 3* a
*t − 1* (2017–2019 para a eleição de 2020; 2021–2023 para a de 2024). As contas do ano da eleição só são
fechadas depois do pleito; usá-las seria **vazamento de informação**, porque o modelo veria dados
que o eleitor ainda não tinha.

| Bloco | Atributo | Definição |
|---|---|---|
| Trabalho | `var_ideb` | Variação do IDEB da rede municipal (anos iniciais) durante o mandato |
| Trabalho | `cresc_pib_pc_rel` | Crescimento do PIB per capita do município menos o crescimento do Brasil no mesmo período (log) |
| Trabalho | `investimento_share` | Investimentos / despesa total (média do mandato) |
| Máquina | `dependencia_transferencias` | Transferências correntes / receitas correntes (média do mandato) |
| Máquina | `pessoal_share` | Despesa com pessoal / despesa total (média do mandato) |
| Máquina | `var_pessoal` | Crescimento da despesa com pessoal entre o 1º e o 3º ano do mandato |
| Máquina | `acel_investimento` | Participação do investimento no ano pré-eleitoral menos a média dos dois anos anteriores |
| Contexto | `log_populacao` | Logaritmo da população (porte do município) |

Usamos **razões e variações** em vez de valores em reais, porque valores nominais mudam com a inflação
entre 2020 e 2024 e distorceriam a comparação entre as duas eleições.

In [10]:
ANOS_MANDATO = {t: [t - 3, t - 2, t - 1] for t in ELEICOES}     # {2020: [2017, 2018, 2019], 2024: [2021, 2022, 2023]}
ANOS_FISCAIS = sorted({a for anos in ANOS_MANDATO.values() for a in anos})
SICONFI = "basedosdados.br_me_siconfi"

def padroniza_id(serie):
    # códigos às vezes chegam como número (1234567.0) e às vezes como texto: padroniza como texto
    return serie.astype(str).str.replace(r"\.0$", "", regex=True)

def coluna(df, chave):
    # devolve a coluna se existir; senão, uma coluna vazia (ano ainda não publicado, por exemplo)
    return df[chave] if chave in df.columns else pd.Series(np.nan, index=df.index)

def norm_txt(s):
    # minúsculas e sem acento, para comparar textos com segurança
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()

# Receitas: receitas correntes e transferências correntes (contas de nível agregado)
rec = consultar(rf'''
SELECT ano, id_municipio, estagio_bd, conta_bd, MAX(valor) AS valor
FROM `{SICONFI}.municipio_receitas_orcamentarias`
WHERE ano IN ({lista(ANOS_FISCAIS)})
  AND (REGEXP_CONTAINS(LOWER(conta_bd), r'^([0-9.]+\s*-\s*)?receitas correntes$')
       OR REGEXP_CONTAINS(LOWER(conta_bd), r'^([0-9.]+\s*-\s*)?transfer\S*ncias correntes$'))
GROUP BY ano, id_municipio, estagio_bd, conta_bd
''')

# Despesas: correntes, de capital, investimentos e pessoal
desp_orc = consultar(rf'''
SELECT ano, id_municipio, estagio_bd, conta_bd, MAX(valor) AS valor
FROM `{SICONFI}.municipio_despesas_orcamentarias`
WHERE ano IN ({lista(ANOS_FISCAIS)})
  AND (REGEXP_CONTAINS(LOWER(conta_bd), r'^([0-9.]+\s*-\s*)?despesas correntes$')
       OR REGEXP_CONTAINS(LOWER(conta_bd), r'^([0-9.]+\s*-\s*)?despesas de capital$')
       OR REGEXP_CONTAINS(LOWER(conta_bd), r'^([0-9.]+\s*-\s*)?investimentos$')
       OR REGEXP_CONTAINS(LOWER(conta_bd), r'^([0-9.]+\s*-\s*)?pessoal e encargos sociais$'))
GROUP BY ano, id_municipio, estagio_bd, conta_bd
''')

print("Estágios das receitas:", rec["estagio_bd"].unique().tolist())
print("Contas de receita encontradas:", rec["conta_bd"].unique().tolist())
print("Estágios das despesas:", desp_orc["estagio_bd"].unique().tolist())
print("Contas de despesa encontradas:", desp_orc["conta_bd"].unique().tolist())

Estágios das receitas: ['Deduções - FUNDEB', 'Deduções - Transferências Constitucionais', 'Outras Deduções da Receita', 'Receitas Brutas Realizadas']
Contas de receita encontradas: ['Transferências Correntes', 'Receitas Correntes']
Estágios das despesas: ['Despesas Empenhadas', 'Despesas Liquidadas', 'Despesas Pagas', 'Inscrição de Restos a Pagar Não Processados', 'Inscrição de Restos a Pagar Processados']
Contas de despesa encontradas: ['Despesas Correntes', 'Pessoal e Encargos Sociais', 'Despesas de Capital', 'Investimentos']


Cada conta aparece em vários **estágios** (receita prevista ou realizada; despesa empenhada, liquidada
ou paga). Usamos a **receita bruta realizada** e a **despesa liquidada** (serviço efetivamente entregue).
O código escolhe o estágio pelo nome e mostra qual foi usado.

In [11]:
def escolher_estagio(estagios, preferencias):
    # Devolve o primeiro estágio cujo nome contém todas as palavras de uma das preferências.
    for palavras in preferencias:
        for e in estagios:
            if all(p in norm_txt(e) for p in palavras):
                return e
    raise ValueError(f"Nenhum estágio compatível em {estagios}")

est_rec = escolher_estagio(rec["estagio_bd"].dropna().unique(), [["brut", "realizad"], ["realizad"]])
est_desp = escolher_estagio(desp_orc["estagio_bd"].dropna().unique(), [["liquidad"], ["empenhad"]])
print("Estágio de receita usado:", est_rec, "| Estágio de despesa usado:", est_desp)

def tabela_larga(df, estagio, nomes):
    # Uma linha por (município, ano) e uma coluna por conta.
    d = df[df["estagio_bd"] == estagio].copy()
    d["id_municipio"] = padroniza_id(d["id_municipio"])
    d["conta"] = d["conta_bd"].map(norm_txt).str.replace(r"^[0-9.]+\s*-\s*", "", regex=True).map(nomes)
    return d.pivot_table(index=["id_municipio", "ano"], columns="conta", values="valor", aggfunc="max").reset_index()

fin = tabela_larga(rec, est_rec, {"receitas correntes": "rec_correntes", "transferencias correntes": "transf_correntes"}).merge(
      tabela_larga(desp_orc, est_desp, {"despesas correntes": "desp_correntes", "despesas de capital": "desp_capital",
                                        "investimentos": "investimentos", "pessoal e encargos sociais": "pessoal"}),
      on=["id_municipio", "ano"], how="outer")

fin["desp_total"] = fin["desp_correntes"].fillna(0) + fin["desp_capital"].fillna(0)
fin["dep_transf"] = fin["transf_correntes"] / fin["rec_correntes"]
fin["pessoal_sh"] = fin["pessoal"] / fin["desp_total"].replace(0, np.nan)
fin["invest_sh"] = fin["investimentos"] / fin["desp_total"].replace(0, np.nan)
print("Municípios com dados fiscais por ano:", fin.dropna(subset=["dep_transf"]).groupby("ano").size().to_dict())

# Indicadores do mandato para cada eleição t
linhas = []
for t, anos in ANOS_MANDATO.items():
    f = fin[fin["ano"].isin(anos)]
    media = f.groupby("id_municipio")[["dep_transf", "pessoal_sh", "invest_sh"]].mean()
    por_ano = f.pivot_table(index="id_municipio", columns="ano", values=["pessoal", "invest_sh"])
    ind = pd.DataFrame({
        "dependencia_transferencias": media["dep_transf"],
        "pessoal_share": media["pessoal_sh"],
        "investimento_share": media["invest_sh"],
        "var_pessoal": coluna(por_ano, ("pessoal", t - 1)) / coluna(por_ano, ("pessoal", t - 3)).replace(0, np.nan) - 1,
        "acel_investimento": coluna(por_ano, ("invest_sh", t - 1))
                             - (coluna(por_ano, ("invest_sh", t - 3)) + coluna(por_ano, ("invest_sh", t - 2))) / 2,
    }).reset_index()
    ind["ano"] = t
    linhas.append(ind)
ind_fiscais = pd.concat(linhas, ignore_index=True)

# limites para erros de declaração (ex.: crescimento de pessoal de 5000%) sem apagar o município
ind_fiscais["var_pessoal"] = ind_fiscais["var_pessoal"].clip(-0.9, 3)
ind_fiscais.describe().T.round(3)

Estágio de receita usado: Receitas Brutas Realizadas | Estágio de despesa usado: Despesas Liquidadas
Municípios com dados fiscais por ano: {np.int64(2017): 5538, np.int64(2018): 5527, np.int64(2019): 5553, np.int64(2021): 5561, np.int64(2022): 5551, np.int64(2023): 5551}


,count,mean,std,min,25%,50%,75%,max
dependencia_transferencias,11132.0,0.861,0.103,0.295,0.814,0.890,0.937,0.996
pessoal_share,11131.0,0.527,0.075,0.189,0.477,0.526,0.574,0.887
investimento_share,11130.0,0.069,0.040,0.003,0.040,0.060,0.089,0.472
var_pessoal,11090.0,0.271,0.232,-0.900,0.132,0.242,0.385,3.000
acel_investimento,11026.0,0.006,0.039,-0.270,-0.015,0.004,0.025,0.279
ano,11132.0,2022.000,2.000,2020.000,2020.000,2022.000,2024.000,2024.000


### População, PIB e IDEB

- **População** (estimativas do IBGE): quando falta o ano exato (por exemplo, no ano do Censo), usamos o
  ano disponível mais próximo.
- **PIB dos municípios** (IBGE): o crescimento é medido em relação ao Brasil, o que elimina o efeito da
  inflação e do ciclo econômico nacional. Se o último ano do mandato ainda não foi publicado, usamos o
  mais recente disponível (o código informa).
- **IDEB** (INEP): rede municipal, anos iniciais do ensino fundamental, que são responsabilidade
  principal das prefeituras. Comparamos o IDEB antes do mandato com o medido durante o mandato
  (2015 → 2019 e 2019 → 2023).

In [12]:
pop = consultar('''
SELECT ano, id_municipio, populacao
FROM `basedosdados.br_ibge_populacao.municipio`
WHERE ano BETWEEN 2015 AND 2024
''')
pib = consultar('''
SELECT ano, id_municipio, pib
FROM `basedosdados.br_ibge_pib.municipio`
WHERE ano >= 2015
''')
ideb = consultar('''
SELECT ano, id_municipio, rede, ensino, anos_escolares, ideb
FROM `basedosdados.br_inep_ideb.municipio`
WHERE ano IN (2015, 2019, 2023)
''')
for df in (pop, pib, ideb):
    df["id_municipio"] = padroniza_id(df["id_municipio"])
    df["ano"] = df["ano"].astype(int)

print("Anos de população:", sorted(pop["ano"].unique()))
print("Anos de PIB:", sorted(pib["ano"].unique()))
print("IDEB - redes:", ideb["rede"].unique().tolist())
print("IDEB - ensino:", ideb["ensino"].unique().tolist(), "| anos escolares:", ideb["anos_escolares"].unique().tolist())

def populacao_no_ano(ano):
    # população de cada município no ano pedido, ou no ano disponível mais próximo
    anos = np.array(sorted(pop["ano"].unique()))
    mais_proximo = int(anos[np.argmin(np.abs(anos - ano))])
    return pop[pop["ano"] == mais_proximo].set_index("id_municipio")["populacao"]

def pib_pc(ano):
    p = pib[pib["ano"] == ano].set_index("id_municipio")["pib"]
    return p / populacao_no_ano(ano)

ultimo_pib = int(pib["ano"].max())
linhas = []
for t in ELEICOES:
    inicio, fim = t - 4, min(t - 1, ultimo_pib)
    print(f"Eleição {t}: crescimento do PIB per capita de {inicio} a {fim}")
    pc_ini, pc_fim = pib_pc(inicio), pib_pc(fim)
    br_ini = pib[pib["ano"] == inicio]["pib"].sum() / populacao_no_ano(inicio).sum()
    br_fim = pib[pib["ano"] == fim]["pib"].sum() / populacao_no_ano(fim).sum()
    cresc = np.log(pc_fim / pc_ini) - np.log(br_fim / br_ini)
    linhas.append(pd.DataFrame({"id_municipio": cresc.index, "ano": t, "cresc_pib_pc_rel": cresc.values,
                                "log_populacao": np.log(populacao_no_ano(t - 1).reindex(cresc.index)).values}))
ind_econ = pd.concat(linhas, ignore_index=True)

# IDEB da rede municipal, anos iniciais
f = ideb[ideb["rede"].map(norm_txt).str.contains("municipal")
         & ideb["anos_escolares"].map(norm_txt).str.contains("inici")]
f = f.groupby(["id_municipio", "ano"])["ideb"].mean().unstack()
ind_ideb = pd.concat([
    pd.DataFrame({"id_municipio": f.index, "ano": 2020, "var_ideb": (coluna(f, 2019) - coluna(f, 2015)).values}),
    pd.DataFrame({"id_municipio": f.index, "ano": 2024, "var_ideb": (coluna(f, 2023) - coluna(f, 2019)).values}),
], ignore_index=True)

print("Municípios com crescimento do PIB:", ind_econ.dropna().groupby("ano").size().to_dict())
print("Municípios com variação do IDEB:", ind_ideb.dropna().groupby("ano").size().to_dict())

Anos de população: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Anos de PIB: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
IDEB - redes: ['estadual', 'municipal', 'federal', 'publica']
IDEB - ensino: ['fundamental', 'medio'] | anos escolares: ['finais (6-9)', 'iniciais (1-5)', 'todos (1-4)']
Eleição 2020: crescimento do PIB per capita de 2016 a 2019
Eleição 2024: crescimento do PIB per capita de 2020 a 2023
Municípios com crescimento do PIB: {2020: 5570, 2024: 5570}
Municípios com variação do IDEB: {2020: 4869, 2024: 5049}


## 10. Base final

Juntamos a base do TSE com os indicadores do mandato pelo código IBGE do município (`id_municipio`).
Valores ausentes são **mantidos como ausentes**: municípios pequenos podem não ter IDEB divulgado, por
exemplo. O tratamento (imputação) é feito no notebook de modelagem, **dentro** da validação, para não
vazar informação do conjunto de teste.

In [13]:
REGIAO = {**dict.fromkeys(["AC", "AM", "AP", "PA", "RO", "RR", "TO"], "Norte"),
          **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
          **dict.fromkeys(["DF", "GO", "MS", "MT"], "Centro-Oeste"),
          **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
          **dict.fromkeys(["PR", "RS", "SC"], "Sul")}

base = base_tse.copy()
base["id_municipio"] = padroniza_id(base["id_municipio"])
base = (base.merge(ind_fiscais, on=["id_municipio", "ano"], how="left")
            .merge(ind_econ, on=["id_municipio", "ano"], how="left")
            .merge(ind_ideb, on=["id_municipio", "ano"], how="left"))
base["regiao"] = base["sigla_uf"].map(REGIAO)
base["log_gasto_campanha_pc"] = np.log1p(base["gasto_campanha"] / np.exp(base["log_populacao"]))

print("Linhas:", len(base), "| duplicatas (ano, município):", base.duplicated(["ano", "id_municipio"]).sum())
print("\nPercentual de valores ausentes por coluna e eleição:")
print((base.drop(columns="ano").isna().groupby(base["ano"]).mean().T * 100).round(1).to_string())

base.to_csv("base_final.csv", index=False)
from google.colab import files
files.download("base_final.csv")

Linhas: 5622 | duplicatas (ano, município): 0

Percentual de valores ausentes por coluna e eleição:
ano                         2020  2024
sigla_uf                     0.0   0.0
id_municipio                 0.0   0.0
id_municipio_tse             0.0   0.0
reeleito                     0.0   0.0
alvo_margem                  0.0   0.0
pct_votos_anterior           0.0   0.0
margem_anterior              0.0   0.0
n_candidatos                 0.0   0.0
troca_partido                0.0   0.0
mesmo_partido_governador     0.0   0.0
coligacao_com_governador     0.0   0.0
n_partidos_coligacao         0.0   0.0
idade                        0.0   0.0
genero_feminino              0.0   0.0
gasto_campanha               0.0   0.0
gasto_campanha_share         0.0   0.0
dependencia_transferencias   0.0   0.0
pessoal_share                0.1   0.0
investimento_share           0.1   0.0
var_pessoal                  0.4   0.3
acel_investimento            1.2   0.5
cresc_pib_pc_rel             0.0   0.0
log

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>